In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "default")
dbutils.widgets.text("landing_volume", "undercover_cars")            # UC volume name
dbutils.widgets.text("object_id", "100079799268376")                        # Page / object id
dbutils.widgets.dropdown("source_mode", "graph_api", ["graph_api", "files"])
dbutils.widgets.text("secret_scope", "facebook")
dbutils.widgets.text("secret_key", "graph_access_token")
dbutils.widgets.text("graph_version", "v21.0")
dbutils.widgets.text("max_pages", "200")                     # safety cap on pagination

In [0]:
# /Volumes/workspace/default/undercover_cars

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Facebook posts ingestion (Bronze -> Silver)
# MAGIC
# MAGIC Two supported sources (set `SOURCE_MODE`):
# MAGIC * `graph_api` – pulls `/{object_id}/feed` from the Meta Graph API (works for a **Page** you administer;
# MAGIC   the Groups API was retired by Meta, so it will not work for ordinary groups).
# MAGIC * `files`     – skips fetching; ingests JSON files you drop into the landing Volume
# MAGIC   (e.g. Meta Content Library / approved research export, or your own "Download your information" dump).
# MAGIC
# MAGIC Flow: fetch -> raw JSONL in UC Volume -> Auto Loader -> `bronze_fb_posts` -> MERGE -> `silver_fb_posts`
# MAGIC
# MAGIC Prerequisites: Unity Catalog, a secret scope holding the Graph API token (`graph_api` mode only).

# COMMAND ----------

dbutils.widgets.text("catalog", "main")
dbutils.widgets.text("schema", "facebook")
dbutils.widgets.text("landing_volume", "landing")            # UC volume name
dbutils.widgets.text("object_id", "")                        # Page / object id
dbutils.widgets.dropdown("source_mode", "graph_api", ["graph_api", "files"])
dbutils.widgets.text("secret_scope", "facebook")
dbutils.widgets.text("secret_key", "graph_access_token")
dbutils.widgets.text("graph_version", "v21.0")
dbutils.widgets.text("max_pages", "200")                     # safety cap on pagination

CATALOG        = dbutils.widgets.get("catalog")
SCHEMA         = dbutils.widgets.get("schema")
VOLUME         = dbutils.widgets.get("landing_volume")
OBJECT_ID      = dbutils.widgets.get("object_id")
SOURCE_MODE    = dbutils.widgets.get("source_mode")
SECRET_SCOPE   = dbutils.widgets.get("secret_scope")
SECRET_KEY     = dbutils.widgets.get("secret_key")
GRAPH_VERSION  = dbutils.widgets.get("graph_version")
MAX_PAGES      = int(dbutils.widgets.get("max_pages"))

LANDING_PATH   = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/posts"
CHECKPOINT     = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/_checkpoints/bronze_fb_posts"
SCHEMA_LOC     = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/_schemas/bronze_fb_posts"
BRONZE_TABLE   = f"{CATALOG}.{SCHEMA}.bronze_fb_posts"
SILVER_TABLE   = f"{CATALOG}.{SCHEMA}.silver_fb_posts"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
dbutils.fs.mkdirs(LANDING_PATH)

# COMMAND ----------
# MAGIC %md ## 1. Fetch from Graph API into landing zone (incremental)

# COMMAND ----------

import json, time, uuid, datetime as dt
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

FIELDS = ",".join([
    "id", "message", "story", "created_time", "updated_time", "permalink_url",
    "status_type", "from{id,name}",
    "attachments{media_type,url,title,description,media}",
    "reactions.summary(total_count).limit(0)",
    "comments.summary(total_count).limit(0)",
    "shares",
])


def _session() -> requests.Session:
    s = requests.Session()
    retry = Retry(total=6, backoff_factor=2,
                  status_forcelist=[429, 500, 502, 503, 504],
                  allowed_methods=["GET"])
    s.mount("https://", HTTPAdapter(max_retries=retry))
    return s


def _last_watermark():
    """Latest created_time already in bronze (epoch seconds) or None for a full load."""
    if not spark.catalog.tableExists(BRONZE_TABLE):
        return None
    row = spark.sql(
        f"SELECT max(unix_timestamp(to_timestamp(created_time, \"yyyy-MM-dd'T'HH:mm:ssZ\"))) AS wm "
        f"FROM {BRONZE_TABLE}"
    ).first()
    return int(row.wm) if row and row.wm else None


def fetch_feed():
    token = dbutils.secrets.get(SECRET_SCOPE, SECRET_KEY)
    params = {"fields": FIELDS, "limit": 100, "access_token": token}
    wm = _last_watermark()
    if wm:
        params["since"] = wm - 3600           # 1h overlap; MERGE dedupes later
    url = f"https://graph.facebook.com/{GRAPH_VERSION}/{OBJECT_ID}/feed"
    http, posts, pages = _session(), [], 0

    while url and pages < MAX_PAGES:
        r = http.get(url, params=params if pages == 0 else None, timeout=60)
        if r.status_code != 200:
            raise RuntimeError(f"Graph API error {r.status_code}: {r.text[:500]}")
        payload = r.json()
        posts.extend(payload.get("data", []))
        url = payload.get("paging", {}).get("next")
        pages += 1
        time.sleep(0.5)                        # be polite / avoid rate limits
    return posts


if SOURCE_MODE == "graph_api":
    if not OBJECT_ID:
        raise ValueError("Set the object_id widget (Page id) for graph_api mode.")
    posts = fetch_feed()
    print(f"Fetched {len(posts)} posts")
    if posts:
        ts = dt.datetime.utcnow().strftime("%Y%m%dT%H%M%S")
        out = f"{LANDING_PATH}/feed_{ts}_{uuid.uuid4().hex[:8]}.jsonl"
        with open(out, "w", encoding="utf-8") as f:
            for p in posts:
                p["_ingested_object_id"] = OBJECT_ID
                f.write(json.dumps(p, ensure_ascii=False) + "\n")
        print(f"Wrote {out}")
else:
    print(f"files mode: expecting JSON/JSONL files in {LANDING_PATH}")

# COMMAND ----------
# MAGIC %md ## 2. Bronze – Auto Loader (raw, append-only)

# COMMAND ----------

from pyspark.sql import functions as F

bronze_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_LOC)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("multiLine", "false")
    .load(LANDING_PATH)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

(
    bronze_stream.writeStream
    .option("checkpointLocation", CHECKPOINT)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(BRONZE_TABLE)
    .awaitTermination()
)

print("Bronze rows:", spark.table(BRONZE_TABLE).count())

# COMMAND ----------
# MAGIC %md ## 3. Silver – flatten, pseudonymise author, upsert by post id

# COMMAND ----------

from delta.tables import DeltaTable
from pyspark.sql.window import Window

bronze = spark.table(BRONZE_TABLE)
cols = set(bronze.columns)


def col_or_null(name, dtype="string"):
    return F.col(name) if name in cols else F.lit(None).cast(dtype)


silver_df = (
    bronze.select(
        col_or_null("id").alias("post_id"),
        col_or_null("message").alias("message"),
        col_or_null("story").alias("story"),
        F.to_timestamp(col_or_null("created_time"), "yyyy-MM-dd'T'HH:mm:ssZ").alias("created_at"),
        F.to_timestamp(col_or_null("updated_time"), "yyyy-MM-dd'T'HH:mm:ssZ").alias("updated_at"),
        col_or_null("permalink_url").alias("permalink_url"),
        col_or_null("status_type").alias("status_type"),
        # GDPR: store a salted hash of the author id, not the name
        F.sha2(F.concat_ws("|", F.lit("fb_salt_v1"), F.col("from.id")), 256).alias("author_hash")
            if "from" in cols else F.lit(None).cast("string").alias("author_hash"),
        (F.col("reactions.summary.total_count").cast("long")
            if "reactions" in cols else F.lit(None).cast("long")).alias("reactions_count"),
        (F.col("comments.summary.total_count").cast("long")
            if "comments" in cols else F.lit(None).cast("long")).alias("comments_count"),
        (F.col("shares.count").cast("long")
            if "shares" in cols else F.lit(None).cast("long")).alias("shares_count"),
        (F.to_json(F.col("attachments")) if "attachments" in cols
            else F.lit(None).cast("string")).alias("attachments_json"),
        F.col("_ingested_at"),
        F.col("_source_file"),
    )
    .where(F.col("post_id").isNotNull())
)

# keep only the latest version of each post
w = Window.partitionBy("post_id").orderBy(F.col("updated_at").desc_nulls_last(),
                                          F.col("_ingested_at").desc())
silver_latest = (silver_df.withColumn("_rn", F.row_number().over(w))
                          .where("_rn = 1").drop("_rn"))

if not spark.catalog.tableExists(SILVER_TABLE):
    silver_latest.write.format("delta").saveAsTable(SILVER_TABLE)
else:
    (
        DeltaTable.forName(spark, SILVER_TABLE).alias("t")
        .merge(silver_latest.alias("s"), "t.post_id = s.post_id")
        .whenMatchedUpdateAll(condition="s.updated_at > t.updated_at OR t.updated_at IS NULL")
        .whenNotMatchedInsertAll()
        .execute()
    )

display(spark.table(SILVER_TABLE).orderBy(F.col("created_at").desc()).limit(20))